# NorthStar - R analytics and charts

## Aim
This notebook uses R to explore the cleaned dataset with summary tables, charts, and a small amount of simple statistical testing.

## Before you run this notebook
Clone the GitHub repository in Colab and run Notebook 1 first.

Notebook 1 creates the cleaned file here:
`/content/assignment/outputs_python/order_delivery_view.csv`

This notebook reads that file directly.

In [ ]:
!pip -q install rpy2

In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R
install_if_missing <- function(package_name) {
  if (!require(package_name, character.only = TRUE)) {
    install.packages(package_name, repos = 'https://cloud.r-project.org')
    library(package_name, character.only = TRUE)
  }
}

install_if_missing('readr')
install_if_missing('dplyr')
install_if_missing('ggplot2')
install_if_missing('tidyr')

In [ ]:
%%R
root_dir <- '/content/assignment'
orders_view <- readr::read_csv(file.path(root_dir, 'outputs_python', 'order_delivery_view.csv'), show_col_types = FALSE)

print(dim(orders_view))
head(orders_view)

In [ ]:
%%R
zone_summary <- orders_view %>%
  group_by(hub_zone_clean) %>%
  summarise(
    total_orders = n(),
    delay_rate = mean(was_late_or_failed, na.rm = TRUE),
    avg_rating = mean(customer_rating_post_delivery, na.rm = TRUE),
    .groups = 'drop'
  ) %>%
  arrange(desc(delay_rate))

service_summary <- orders_view %>%
  group_by(service_type) %>%
  summarise(
    total_orders = n(),
    avg_order_value = mean(order_value, na.rm = TRUE),
    complaint_rate = mean(has_complaint, na.rm = TRUE),
    .groups = 'drop'
  ) %>%
  arrange(desc(complaint_rate))

rating_by_complaint <- orders_view %>%
  mutate(complaint_group = ifelse(has_complaint == 1, 'Complaint', 'No complaint')) %>%
  group_by(complaint_group) %>%
  summarise(
    avg_rating = mean(customer_rating_post_delivery, na.rm = TRUE),
    total_orders = n(),
    .groups = 'drop'
  )

zone_summary
service_summary
rating_by_complaint

In [ ]:
%%R
ggplot(zone_summary, aes(x = reorder(hub_zone_clean, delay_rate), y = delay_rate)) +
  geom_col(fill = 'steelblue') +
  coord_flip() +
  labs(
    title = 'Delay rate by hub zone',
    x = 'Hub zone',
    y = 'Delay or fail rate'
  ) +
  theme_minimal()

In [ ]:
%%R
rating_test_data <- orders_view %>%
  filter(!is.na(customer_rating_post_delivery))

complaint_rating_test <- t.test(
  customer_rating_post_delivery ~ has_complaint,
  data = rating_test_data
)

complaint_rating_test

### Small statistical point

This simple test checks whether ratings differ between orders with complaints and orders without complaints.

You do not need heavy theory here. A short explanation is enough: if the averages are clearly different, complaint cases are also customer experience cases.

### Interpretation of chart 1

This chart shows that some zones perform much worse than others. If Airport and Central are near the top, managers should treat them as priority areas.

This matters because the business problem is not random. It seems to be concentrated in a few places.

In [ ]:
%%R
ggplot(service_summary, aes(x = reorder(service_type, complaint_rate), y = complaint_rate)) +
  geom_col(fill = 'tomato') +
  coord_flip() +
  labs(
    title = 'Complaint rate by service type',
    x = 'Service type',
    y = 'Complaint rate'
  ) +
  theme_minimal()

In [ ]:
%%R
ggplot(rating_by_complaint, aes(x = complaint_group, y = avg_rating, fill = complaint_group)) +
  geom_col() +
  labs(
    title = 'Average rating by complaint group',
    x = 'Complaint group',
    y = 'Average customer rating'
  ) +
  theme_minimal() +
  guides(fill = 'none')

### Interpretation of chart 3

This chart helps show that complaint cases usually come with lower customer ratings. That gives more support to the idea that complaints and service quality are closely linked.

It also adds a simple statistical story to the visual work.

### Interpretation of chart 2

Retail appears to create the highest complaint rate. That means customer friction is linked not only to place, but also to service type.

If one service type keeps appearing at the top, it is a good target for process review and service redesign.

## Final findings

- Airport and Central look like the main weak areas in the network.
- Retail stands out as a service type with more complaint pressure.
- Ratings are lower when complaints are present, so customer experience and service failure are clearly linked.
- The R notebook adds value because it turns the cleaned file into simple visuals and a small amount of statistical evidence.
- This supports the case study goal of finding real business problems, not only describing the data.